In [ ]:
import duckdb as db
import pandas as pd


In [ ]:
# 1. Load data
fex = db.query("SELECT * FROM './original_data/fex.parquet'").df()

In [ ]:
# Definying pools, sampling fraction and apparent prevalence
apv = db.query( """
WITH raw_data AS (
    SELECT 
        farm_id,
        field_exame_id,
        CAST(field_exame_date AS DATE) as exam_date,
        pos,
        nex,
        Ntotal
    FROM fex
),
sorted_fex AS (
    SELECT 
        *,
        LAG(exam_date) OVER (PARTITION BY farm_id ORDER BY exam_date) as prev_date
    FROM raw_data
),
pool_starts AS (
    SELECT 
        *,
        CASE 
            WHEN prev_date IS NULL THEN 1
            WHEN date_diff('day', prev_date, exam_date) > 90 THEN 1 
            ELSE 0 
        END as is_new_pool
    FROM sorted_fex
),
fex_with_pool_id AS (
    SELECT 
        *,
        SUM(is_new_pool) OVER (PARTITION BY farm_id ORDER BY exam_date) as pool_num
    FROM pool_starts
),
aparent_prevalence_pool AS (
    SELECT 
        farm_id,
        pool_num,
        MIN(exam_date) as round_start,
        MAX(exam_date) as round_end,
        SUM(pos) as total_pos,
        SUM(nex) as total_nex,
        MAX(Ntotal) as max_herd_size
    FROM fex_with_pool_id
    GROUP BY farm_id, pool_num
)
SELECT 
    farm_id,
    round_start,
    round_end,
    total_pos,
    total_nex,
    max_herd_size as avg_herd_size,
    (total_pos::FLOAT / NULLIF(total_pos + total_nex, 0)) as apparent_prevalence,
    LEAST(1.0, (total_nex::FLOAT / NULLIF(max_herd_size, 0))) as sampling_fraction,
    CASE 
        WHEN total_pos > 0 THEN 'Evidence of Infection'
        WHEN (total_nex::FLOAT / NULLIF(max_herd_size, 0)) >= 0.80 OR total_nex >= 60 THEN 'Gold Standard'
        WHEN (total_nex::FLOAT / NULLIF(max_herd_size, 0)) >= 0.30 OR total_nex >= 20 THEN 'Acceptable'
        ELSE 'Under-sampled'
    END AS sampling_confidence
FROM aparent_prevalence_pool
ORDER BY farm_id, round_start;
"""
).df()


In [2]:
db.query("SELECT * FROM apv").to_parquet("apv.parquet")